# 基于欧氏域对齐与轻量时空网络的跨被试运动想象（MI）解码

本 Notebook 在 **Google Colab** 中完成 BCI Competition IV-2a（22 导四分类）数据集自动下载缓存、无监督欧氏对齐（EA）、FBCSP+黎曼切空间、ShallowConvNet、EEGNet、EEG-Conformer（时空卷积+多头自注意力）的 **严格留一被试交叉验证（LOSO）消融实验** 及 Grad-CAM 脑电极重要性分析，所有结果自动持久化至 Google Drive。

- **推荐运行时**：`代码执行程序 -> 更改运行时类型 -> T4 GPU`
- **云端项目路径**：`/content/drive/MyDrive/BCI_Portfolio/02_BCI_IV_2a_MI_EA_Conformer`

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/BCI_Portfolio/02_BCI_IV_2a_MI_EA_Conformer')
assert PROJECT_DIR.exists(), f'未找到项目目录: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('工作目录:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
import torch, mne, moabb, pyriemann
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | MOABB: {moabb.__version__}')

In [ ]:
# 自动下载并缓存 BCI Competition IV-2a 全部 9 名被试数据到 Google Drive
!python scripts/download_dataset.py

In [ ]:
from src.utils import load_config, set_seed, setup_logger
from src.evaluation import run_loso_benchmark

RUN_MODE = 'full'  # 'full' = 完整 9 折 LOSO（全部 9 名被试）；'quick' = 前 3 折快速验证

cfg = load_config(PROJECT_DIR / 'configs' / 'config.yaml')
cfg['project']['data_dir'] = str(PROJECT_DIR / cfg['project']['data_dir'])
cfg['project']['results_dir'] = str(PROJECT_DIR / cfg['project']['results_dir'])

set_seed(int(cfg['project']['seed']))
setup_logger('mi_conformer', log_file=PROJECT_DIR / 'results' / 'run.log')

targets = cfg['dataset']['subjects'] if RUN_MODE == 'full' else cfg['dataset']['quick_subjects']
summary = run_loso_benchmark(target_subjects=targets, cfg=cfg)

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

results_dir = PROJECT_DIR / 'results'
print('云端已保存结果文件:', sorted(os.listdir(results_dir)))
md_path = results_dir / 'benchmark_report.md'
if md_path.exists():
    display(Markdown(md_path.read_text(encoding='utf-8')))
if (results_dir / 'gradcam_channel_importance.csv').exists():
    print('\n--- 22 导电极 Grad-CAM 重要性全表 ---')
    display(pd.read_csv(results_dir / 'gradcam_channel_importance.csv').head(10))